# Multiagent Pipeline

In [29]:
# Facoltativo: installa dipendenze se ti servono in un ambiente pulito
# %pip install -r requirements.txt

# Configuration

In [30]:
import os
from dotenv import load_dotenv

load_dotenv()

LM_STUDIO_BASE_URL = "https://api.mistral.ai/v1"
LM_STUDIO_API_KEY  = os.getenv("MISTRAL_API_KEY", "")
LM_STUDIO_MODEL    = "mistral-small-latest"

# Paths 
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
ALLARMI_CSV = os.path.join(RAW_DIR, "ALLARMI.csv")
TIPOLOGIA_CSV = os.path.join(RAW_DIR, "TIPOLOGIA_VIAGGIATORE.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
FINDINGS_JSON = os.path.join(OUTPUT_DIR, "findings.json")

# Outlier Detection 
DEFAULT_OUTLIER_ALGORITHM = "IsolationForest"
ISOLATION_FOREST_CONTAMINATION = 0.05
LOF_N_NEIGHBORS = 20
ZSCORE_THRESHOLD = 3.0

# Risk Thresholds 
ALERT_RATE_MULTIPLIER = 3.0


In [31]:
import re
import json
import math
from collections import Counter
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd

# NOTE: profiling helpers are plain Python functions here so they can be called directly.


# =============================================================================
# JSON helpers
# =============================================================================

def _to_native(value: Any) -> Any:
    """Convert pandas/numpy objects into native JSON-serializable Python types."""
    if isinstance(value, dict):
        return {str(k): _to_native(v) for k, v in value.items()}
    if isinstance(value, (np.ndarray, list, tuple)):
        return [_to_native(v) for v in value]
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if math.isnan(float(value)):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def _safe_load_json(path: str) -> dict:
    """Load a JSON file safely; return an empty dict if missing/corrupted."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def _safe_write_json(path: str, data: dict) -> None:
    """Write JSON deterministically."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_native(data), f, ensure_ascii=False, indent=2)


# =============================================================================
# Missing values helpers
# =============================================================================

DEFAULT_MISSING_TOKENS = {
    "", " ", "na", "n/a", "null", "none", "nan", "unknown", "undefined",
    "-", "--", "not available", "missing", "n.d.", "nd"
}


def _normalize_missing_mask(series: pd.Series, missing_tokens: Optional[set[str]] = None) -> pd.Series:
    """
    Return a boolean mask for missing values, including semantic placeholders.
    """
    missing_tokens = missing_tokens or DEFAULT_MISSING_TOKENS

    native_missing = series.isna()

    as_str = series.astype("string").str.strip().str.lower()
    semantic_missing = as_str.isin(missing_tokens)

    return native_missing | semantic_missing.fillna(False)


def _non_missing_series(series: pd.Series, missing_tokens: Optional[set[str]] = None) -> pd.Series:
    mask = _normalize_missing_mask(series, missing_tokens=missing_tokens)
    return series.loc[~mask]


# =============================================================================
# Basic dataframe-level tools
# =============================================================================
def get_dataframe_shape(df: pd.DataFrame) -> dict:
    """Return dataframe shape."""
    rows, cols = df.shape
    return {"rows": int(rows), "columns": int(cols)}
def get_dataframe_nulls(df: pd.DataFrame) -> dict:
    """Return null counts per column, including semantic missing tokens."""
    result = {}
    for col in df.columns:
        mask = _normalize_missing_mask(df[col])
        result[col] = {
            "null_count": int(mask.sum()),
            "non_null_count": int((~mask).sum())
        }
    return result
def get_dataframe_dtypes(df: pd.DataFrame) -> dict:
    """Return pandas dtype per column."""
    return {col: str(dtype) for col, dtype in df.dtypes.items()}
def get_missing_percentages(df: pd.DataFrame) -> dict:
    """Return missing percentage per column."""
    total_rows = max(len(df), 1)
    result = {}
    for col in df.columns:
        mask = _normalize_missing_mask(df[col])
        result[col] = {
            "missing_percentage": round(float(mask.mean() * 100), 2),
            "missing_ratio": round(float(mask.sum() / total_rows), 4)
        }
    return result


# =============================================================================
# Format detection
# =============================================================================

_PATTERN_LIBRARY = {
    "integer": re.compile(r"^[+-]?\d+$"),
    "float": re.compile(r"^[+-]?\d+[.,]\d+$"),
    "date_iso": re.compile(r"^\d{4}-\d{2}-\d{2}$"),
    "date_eu": re.compile(r"^\d{2}/\d{2}/\d{4}$"),
    "date_us": re.compile(r"^\d{2}-\d{2}-\d{4}$"),
    "datetime_iso": re.compile(r"^\d{4}-\d{2}-\d{2}[T\s]\d{2}:\d{2}(:\d{2})?$"),
    "email": re.compile(r"^[A-Z0-9._%+\-]+@[A-Z0-9.\-]+\.[A-Z]{2,}$", re.I),
    "uuid": re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$", re.I),
    "alphanumeric_code": re.compile(r"^[A-Z0-9\-_\/]+$", re.I),
    "capitalized_words": re.compile(r"^[A-ZÀ-ÖØ-Ý][a-zà-öø-ÿ]+(?:[\s\-'][A-ZÀ-ÖØ-Ý]?[a-zà-öø-ÿ]+)*$"),
}


def _value_format_label(value: Any) -> str:
    if value is None:
        return "missing"

    s = str(value).strip()

    if s == "":
        return "empty"

    # SOLO NUMERI
    if s.isdigit():
        return "numeric_string"

    # FLOAT (tipo 12.34)
    try:
        float(s)
        return "float_string"
    except:
        pass

    # SOLO LETTERE
    if s.isalpha():
        return "alpha_string"

    # LETTERE + NUMERI (vero alfanumerico)
    has_alpha = any(c.isalpha() for c in s)
    has_digit = any(c.isdigit() for c in s)

    if has_alpha and has_digit:
        return "alphanumeric_code"

    # fallback
    return "generic_string"


def detect_dominant_format(series: pd.Series) -> dict:
    """
    Detect the single most frequent value format in a column.
    """
    clean = _non_missing_series(series)
    if clean.empty:
        return {
            "dominant_format": None,
            "dominant_format_share": 0.0
        }

    labels = clean.map(_value_format_label)
    counts = labels.value_counts(dropna=False)
    total = int(counts.sum())

    dominant_format = str(counts.index[0])
    dominant_share = round(float(counts.iloc[0] / total * 100), 2)

    return {
        "dominant_format": dominant_format,
        "dominant_format_share": dominant_share
    }


# =============================================================================
# Semantic inference
# =============================================================================

def _try_numeric(series: pd.Series) -> pd.Series:
    """
    Convert values to numeric where possible, supporting comma decimals.
    """
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    s = series.astype("string").str.strip()
    s = s.str.replace(".", "", regex=False)  # optional thousands separator cleanup
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def _infer_numeric_semantics(series: pd.Series) -> dict:
    numeric = _try_numeric(series)
    numeric = numeric.dropna()

    if numeric.empty:
        return {
            "semantic_family": "numeric",
            "semantic_type": "unknown_numeric",
            "confidence": 0.0,
            "evidence": []
        }

    n_unique = int(numeric.nunique())
    min_v = float(numeric.min())
    max_v = float(numeric.max())
    integer_like_share = float((numeric % 1 == 0).mean())

    evidence = []

    # Binary / flag
    unique_values = sorted(numeric.dropna().unique().tolist())
    if set(unique_values).issubset({0, 1}):
        evidence.append("Values are only 0 and 1.")
        return {
            "semantic_family": "numeric",
            "semantic_type": "binary_flag",
            "confidence": 0.98,
            "evidence": evidence
        }

    # Percentage / rate
    if min_v >= 0 and max_v <= 100:
        evidence.append("Values fall between 0 and 100.")
        if integer_like_share < 0.95:
            evidence.append("Values are not mostly integers.")
        return {
            "semantic_family": "numeric",
            "semantic_type": "percentage_or_rate",
            "confidence": 0.78,
            "evidence": evidence
        }

    # Year
    if integer_like_share > 0.98 and min_v >= 1900 and max_v <= 2100:
        evidence.append("Values are integer-like and fall in a plausible year range.")
        return {
            "semantic_family": "numeric",
            "semantic_type": "year",
            "confidence": 0.95,
            "evidence": evidence
        }

    # Count
    if min_v >= 0 and integer_like_share > 0.98:
        evidence.append("Values are non-negative integers.")
        return {
            "semantic_family": "numeric",
            "semantic_type": "count_or_quantity",
            "confidence": 0.82,
            "evidence": evidence
        }

    # Identifier-like numeric
    if integer_like_share > 0.98 and n_unique / max(len(numeric), 1) > 0.9:
        evidence.append("Values are almost unique and integer-like.")
        return {
            "semantic_family": "numeric",
            "semantic_type": "numeric_identifier",
            "confidence": 0.76,
            "evidence": evidence
        }

    # Continuous measure
    if integer_like_share < 0.8:
        evidence.append("Values behave like continuous measurements.")
        return {
            "semantic_family": "numeric",
            "semantic_type": "continuous_measure",
            "confidence": 0.8,
            "evidence": evidence
        }

    return {
        "semantic_family": "numeric",
        "semantic_type": "generic_numeric_value",
        "confidence": 0.6,
        "evidence": ["No stronger numeric pattern detected."]
    }


def _infer_string_semantics(series: pd.Series) -> dict:
    clean = _non_missing_series(series)
    if clean.empty:
        return {
            "semantic_family": "string",
            "semantic_type": "unknown_text",
            "confidence": 0.0,
            "evidence": []
        }

    s = clean.astype("string").str.strip()
    lower = s.str.lower()
    n = len(s)
    unique_ratio = float(s.nunique() / max(n, 1))
    avg_words = float(s.str.split().str.len().mean())
    avg_len = float(s.str.len().mean())

    def share(pattern: re.Pattern) -> float:
        return float(s.str.match(pattern, na=False).mean())

    email_share = share(_PATTERN_LIBRARY["email"])
    uuid_share = share(_PATTERN_LIBRARY["uuid"])
    code_share = share(_PATTERN_LIBRARY["alphanumeric_code"])
    name_share = share(_PATTERN_LIBRARY["capitalized_words"])

    evidence = []

    if email_share > 0.8:
        evidence.append("Most values match an email pattern.")
        return {
            "semantic_family": "string",
            "semantic_type": "email",
            "confidence": round(email_share, 2),
            "evidence": evidence
        }

    if uuid_share > 0.8:
        evidence.append("Most values match a UUID pattern.")
        return {
            "semantic_family": "string",
            "semantic_type": "uuid",
            "confidence": round(uuid_share, 2),
            "evidence": evidence
        }

    # Date-like strings
    format_info = detect_dominant_format(series)
    if format_info["dominant_format"] in {"date_iso", "date_eu", "date_us", "datetime_iso", "date_or_datetime"}:
        evidence.append("Most values follow a date/datetime format.")
        return {
            "semantic_family": "string",
            "semantic_type": "date_like_string",
            "confidence": 0.9,
            "evidence": evidence
        }

    # Code / identifier
    if unique_ratio > 0.85 and code_share > 0.8 and avg_words <= 1.5:
        evidence.append("Values are mostly unique, compact, and code-like.")
        return {
            "semantic_family": "string",
            "semantic_type": "identifier_or_code",
            "confidence": 0.84,
            "evidence": evidence
        }

    # Name-like
    if name_share > 0.65 and avg_words <= 4:
        evidence.append("Many values look like person/place/organization names.")
        return {
            "semantic_family": "string",
            "semantic_type": "name_like",
            "confidence": round(max(name_share, 0.7), 2),
            "evidence": evidence
        }

    # Low-cardinality category
    if unique_ratio < 0.2 and avg_words <= 3:
        evidence.append("Low cardinality suggests a categorical label.")
        return {
            "semantic_family": "string",
            "semantic_type": "category_label",
            "confidence": 0.86,
            "evidence": evidence
        }

    # Free text / descriptive field
    if avg_words >= 4 or avg_len >= 30:
        evidence.append("Values are relatively long and descriptive.")
        return {
            "semantic_family": "string",
            "semantic_type": "descriptive_text",
            "confidence": 0.82,
            "evidence": evidence
        }

    return {
        "semantic_family": "string",
        "semantic_type": "generic_text",
        "confidence": 0.58,
        "evidence": ["No stronger text pattern detected."]
    }
def infer_column_semantics(series: pd.Series) -> dict:
    """
    Infer the semantic role of a column:
    - numeric: binary flag, year, count, measure, percentage, identifier...
    - string: identifier, category, name, email, free text...
    """
    clean = _non_missing_series(series)
    if clean.empty:
        return {
            "semantic_family": "unknown",
            "semantic_type": "empty_or_missing",
            "confidence": 0.0,
            "evidence": []
        }

    numeric = _try_numeric(clean)
    numeric_parse_ratio = float(numeric.notna().mean())

    if pd.api.types.is_numeric_dtype(clean) or numeric_parse_ratio >= 0.9:
        return _infer_numeric_semantics(clean)

    return _infer_string_semantics(clean)


# =============================================================================
# Column profiling
# =============================================================================

def profile_single_column(df: pd.DataFrame, column_name: str) -> dict:
    """
    Build a complete profile for one column.
    """
    series = df[column_name]
    missing_mask = _normalize_missing_mask(series)
    non_missing = series.loc[~missing_mask]

    sample_values = [_to_native(v) for v in non_missing.head(5).tolist()]

    profile = {
        "column_name": column_name,
        "dtype": str(series.dtype),
        "row_count": int(len(series)),
        "null_count": int(missing_mask.sum()),
        "non_null_count": int((~missing_mask).sum()),
        "missing_percentage": round(float(missing_mask.mean() * 100), 2),
        "n_unique_non_null": int(non_missing.nunique(dropna=True)),
        "unique_ratio_non_null": round(
            float(non_missing.nunique(dropna=True) / max(len(non_missing), 1)), 4
        ),
        "sample_values": sample_values,
        "dominant_format": detect_dominant_format(series),
        "semantic_inference": infer_column_semantics(series),
    }

    # Add numerical summary if the column is numeric or mostly numeric
    numeric = _try_numeric(non_missing)
    if len(non_missing) > 0 and numeric.notna().mean() >= 0.9:
        numeric = numeric.dropna()
        if not numeric.empty:
            profile["numeric_summary"] = {
                "min": _to_native(numeric.min()),
                "max": _to_native(numeric.max()),
                "mean": _to_native(round(float(numeric.mean()), 4)),
                "median": _to_native(round(float(numeric.median()), 4)),
                "std": _to_native(round(float(numeric.std(ddof=1)), 4)) if len(numeric) > 1 else None,
            }

    return _to_native(profile)
def profile_all_columns(df: pd.DataFrame) -> dict:
    """
    Build a full profile for all columns of a dataframe.
    """
    return {
        col: profile_single_column(df, col)
        for col in df.columns
    }


# =============================================================================
# LLM description
# =============================================================================

def generate_column_description_with_llm(
    column_profile: dict,
    llm_callable: Optional[Callable[[str], str]] = None
) -> str:
    """
    Generate a short business-style description for the column.
    llm_callable must be a function that receives a prompt and returns text.
    """
    if llm_callable is None:
        semantic = column_profile.get("semantic_inference", {})
        dominant = column_profile.get("dominant_format", {})
        return (
            f"This column appears to store {semantic.get('semantic_type', 'unknown values')} "
            f"with dtype {column_profile.get('dtype')}. The dominant observed format is "
            f"{dominant.get('dominant_format')} and the missing rate is "
            f"{column_profile.get('missing_percentage')}%."
        )

    prompt = f"""
You are profiling a dataset column.

Write a brief description in 1-2 sentences.
Be precise, compact, and business-friendly.
Do not invent facts that are not present in the profile.

Column profile:
{json.dumps(column_profile, ensure_ascii=False, indent=2)}
""".strip()

    try:
        response = llm_callable(prompt)
        return str(response).strip()
    except Exception as e:
        return f"LLM description failed: {e}"


# =============================================================================
# Main orchestration
# =============================================================================

def build_dataset_column_profiles(
    df: pd.DataFrame,
    dataset_name: str,
    output_json_path: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    findings_root_key: str = "column_profiles"
) -> dict:
    """
    Build a JSON profile for every column in the dataframe and save it.

    Structure:
    {
      findings_root_key: {
        dataset_name: {
          "dataset_summary": {...},
          "columns": {
             "col_a": {..., "llm_description": "..."},
             ...
          }
        }
      }
    }
    """
    findings = _safe_load_json(output_json_path)

    dataset_summary = {
        "dataset_name": dataset_name,
        "shape": get_dataframe_shape(df),
        "dtypes": get_dataframe_dtypes(df),
    }

    columns_payload = {}
    for col in df.columns:
        col_profile = profile_single_column(df, col)
        col_profile["llm_description"] = generate_column_description_with_llm(
            col_profile,
            llm_callable=llm_callable
        )
        columns_payload[col] = col_profile

    findings.setdefault(findings_root_key, {})
    findings[findings_root_key][dataset_name] = {
        "dataset_summary": dataset_summary,
        "columns": columns_payload
    }

    _safe_write_json(output_json_path, findings)
    return findings[findings_root_key][dataset_name]


# =============================================================================
# Optional convenience function from CSV path
# =============================================================================

def build_dataset_column_profiles_from_csv(
    csv_path: str,
    output_json_path: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    dataset_name: Optional[str] = None,
    read_csv_kwargs: Optional[dict] = None,
) -> dict:
    read_csv_kwargs = read_csv_kwargs or {}
    dataset_name = dataset_name or os.path.splitext(os.path.basename(csv_path))[0]

    df = pd.read_csv(csv_path, **read_csv_kwargs)
    return build_dataset_column_profiles(
        df=df,
        dataset_name=dataset_name,
        output_json_path=output_json_path,
        llm_callable=llm_callable,
    )

# Tool REPL

In [32]:
import sys
import io
import traceback
from typing import Annotated
from langchain_core.tools import tool


class PythonREPL:
    """Executes Python code and captures output."""
    def __init__(self):
        self.globals = {}

    def run(self, code: str) -> str:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout = mystdout = io.StringIO()
        sys.stderr = mystderr = io.StringIO()
        try:
            exec(code, self.globals)
            output = mystdout.getvalue()
            errors = mystderr.getvalue()
            if errors:
                output += f"\nSTDERR:\n{errors}"
            return output if output else "(OK, no output)"
        except Exception:
            return f"Error:\n{traceback.format_exc()}"
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr


_repl = PythonREPL()


# Prompts
Natural language prompts for each pipeline task.
The LLM receives these and generates Python code autonomously.

In [33]:
# ── Findings helper ──────────────────────────────────────────────────────────

def _findings_guidance(task_key: str, extra_notes: str = "") -> str:
    base = (
        f"Maintain a shared findings JSON at '{FINDINGS_JSON}'. "
        f"At the start, attempt to load it; if missing, empty, invalid, or corrupted, initialize an empty dict instead of failing. "
        f"Store new information under the key '{task_key}' while preserving existing keys for other tasks. "
        f"Use concise, machine-readable fields with only native Python JSON-serializable types such as dict, list, str, int, float, bool, or null. "
        f"Convert pandas and numpy values to native Python types before saving. "
        f"Convert tuples to lists before saving. "
        f"After completing the task, update the entry for '{task_key}' and write the full JSON back by overwriting the file. "
    )
    if extra_notes:
        base += extra_notes
    return base


# ── Task 1: Clean the dataset ─────────────────────────────────────────────────

def _build_structural_cleaning_prompt(input_path, output_path):
    return (
        f"You are a data cleaning agent for tabular datasets used in downstream merge and anomaly detection tasks. "
        f"Load the dataset at '{input_path}' and produce a cleaned version saved to '{output_path}'. "
        f"Work autonomously and infer the required Python libraries. "
        f"First inspect the raw schema and print the original shape and original column names. "
        f"Then standardize column names to lowercase snake_case. "
        f"Apply column-name normalization at the token or word level, not at the individual-character level. "
        f"Do not insert separators between consecutive characters when a name is already a single continuous token written in uppercase or lowercase. "
        f"Convert naming styles consistently by identifying word boundaries, replacing non-alphanumeric separators with underscores, collapsing repeated underscores, and trimming leading or trailing underscores. "
        f"Immediately after renaming, resolve duplicate column names deterministically by occurrence order. "
        f"When duplicate column labels are present, rebuild the full ordered list of column names so that each occurrence gets a unique stable name. "
        f"Do not use ambiguous renaming by duplicated label alone. "
        f"Do not continue until column names are truly unique and explicitly verified. "
        f"After the schema is safe, clean values column by column. "
        f"The cleaned dataset must have all column names in lowercase snake_case and all textual values in lowercase. "
        f"This is a mandatory output requirement for the cleaned file. "
        f"For every column, apply text normalization to all values that are textual, regardless of how the column type is internally represented. "
        f"Do not rely on a specific column type such as 'object' to detect text. "
        f"Instead, treat any value that behaves like text as textual content and normalize it. "
        f"Convert all textual values to lowercase, remove outer whitespace, and normalize repeated internal whitespace. "
        f"Preserve null values as null. "
        f"If a column contains both text and non-text values, normalize the textual cells to lowercase and leave the non-text cells unchanged. "
        f"Do not skip lowercase normalization for textual values because of mixed content in the same column. "
        f"Do not change the semantic type of non-text values only for convenience. "
        f"The lowercase normalization of textual values must be applied consistently across all datasets, even if different internal data types are used. "
        f"Use one shared and globally defined set of missing-value patterns before any column-level cleaning logic starts. "
        f"Do not define cleaning rules or missing-value rules only inside conditional branches if they are reused elsewhere. "
        f"Keep the cleaning flow structurally simple and deterministic across datasets. "
        f"Use the same cleaning strategy for every dataset of this task category, rather than inventing dataset-specific logic unless strictly necessary for loadability. "
        f"Avoid branch-specific variables that may be unavailable in other branches. "
        f"The generated code must run top-to-bottom without relying on variables defined only inside a conditional block. "
        f"Detect values that semantically represent missing, undefined, or non-informative entries by analyzing the column content, and convert them into proper missing values where appropriate. "
        f"Then remove duplicate rows. "
        f"Handle missing values conservatively and preserve missingness when imputation could distort downstream anomaly detection. "
        f"Before saving, validate that the dataframe is non-empty and has unique column names. "
        f"Do not save any output if those checks fail. "
        f"Print only a short summary with: original shape, final shape, final column names, duplicate-name collisions handled, and duplicate rows removed. "
    )

def _build_data_prompt_1():
    return (
        _build_structural_cleaning_prompt(
            ALLARMI_CSV,
            f"{OUTPUT_DIR}/allarmi_clean.csv"
        )
        + _findings_guidance(
            "data_loading_allarmi",
            "Capture shape_before, shape_after, columns_final, duplicate_name_collisions, duplicate_rows_removed. "
        )
    )

def _build_data_prompt_2():
    return (
        _build_structural_cleaning_prompt(
            TIPOLOGIA_CSV,
            f"{OUTPUT_DIR}/tipologia_clean.csv"
        )
        + _findings_guidance(
            "data_loading_tipologia",
            "Capture shape_before, shape_after, columns_final, duplicate_name_collisions, duplicate_rows_removed. "
        )
    )


# ── Task 2: Normalize the dataset ─────────────────────────────────────────────

def _build_semantic_normalization_prompt(input_path, output_path, findings_key):
    return (
        f"You are a semantic normalization agent for tabular datasets already cleaned at schema level. "
        f"Load the dataset at '{input_path}' and produce a semantically refined version saved to '{output_path}'. "
        f"Work autonomously and infer the required Python libraries. "
        f"The dataset already has a safe schema, so do not rename, reorder, merge, split, or drop columns unless strictly necessary. "
        f"Focus only on intra-column semantic consistency. "
        f"For each column, inspect non-null values and infer the dominant representation. "
        f"Ignore missing or non-informative values when inferring patterns. "
        f"Normalize only high-confidence minority variants. "
        f"Do not collapse distinct categories. "
        f"Preserve uncertain or difform values and report them. "
        f"Before saving, validate that the dataframe is non-empty and still loadable. "
        f"The dataset must be loaded, processed, and saved within the same execution flow. "
        f"All steps must run automatically without requiring any explicit invocation or entry point. "
        f"If validation passes, write the output file immediately using to_csv. "
        f"Do not check whether the output file already exists before writing it. "
        f"The output file must always be created during execution if the dataframe is valid. "
        f"Save the semantically refined dataset to '{output_path}' without index. "
        + _findings_guidance(
            findings_key,
            "Include shape_before, shape_after, normalization_mappings, preserved_difform_values. "
        )
    )


# ── Task 3: Merge ─────────────────────────────────────────────────────────────

def _build_merge_prompt():
    return (
        f"You are a data integration agent responsible for combining two cleaned tabular datasets into a single unified dataset for downstream anomaly detection. "
        f"Load the dataset at '{OUTPUT_DIR}/allarmi_clean.csv' and the dataset at '{OUTPUT_DIR}/tipologia_clean.csv' using pandas. "
        f"Work autonomously and infer the required Python libraries. "
        f"First, inspect both dataframes independently: print their shapes and column names. "
        f"Then identify the common columns between the two dataframes and print them explicitly before proceeding. "
        f"Merge the two dataframes on the common columns using an outer join to preserve all records from both sources. "
        f"After merging, ensure that no duplicate columns remain in the result. "
        f"Do not drop or rename any column unless it is a confirmed structural duplicate introduced by the merge. "
        f"Do not perform any imputation, encoding, or transformation on the merged data. "
        f"Preserve the original values and types from both sources as-is. "
        f"Before saving, validate that the merged dataframe is non-empty and has unique column names. "
        f"Do not save any output if those checks fail. "
        f"Save the merged dataset to '{OUTPUT_DIR}/merged_data.csv' without index. "
        f"Ensure that the dataset is loaded, processed, and saved within the same execution flow. "
        f"Avoid defining execution entry points or structures that require explicit invocation. "
        f"Assume that the code will be executed exactly as written, so all steps must run immediately. "
        f"Print only a short summary with: shape of allarmi_clean, shape of tipologia_clean, common columns used for merge, final merged shape, and duplicate columns removed. "
        + _findings_guidance(
            "merge",
            "Record shapes_allarmi_tipologia, common_columns, merged_shape, duplicate_columns_removed. "
        )
    )


# ── Task 4: Group by route ────────────────────────────────────────────────────

def _build_baseline_prompt():
    return (
        f"You are a data aggregation agent responsible for building a route-level summary dataset to be used as baseline input for downstream anomaly detection. "
        f"Load the dataset at '{OUTPUT_DIR}/merged_data.csv'. "
        f"Work autonomously and infer the required Python libraries. "
        f"First, inspect the loaded dataframe: print its shape and column names before proceeding. "
        f"Create a 'route' column by combining the values of 'areoporto_partenza' and 'areoporto_arrivo'. "
        f"Do not drop the original airport columns after creating the route column. "
        f"Build the aggregation strategy for every column except 'route'. "
        f"Do not hardcode column names when defining the aggregation strategy. "
        f"Determine whether a column is numeric using pandas-native type inspection, not numpy subtype checks. "
        f"For numeric columns, aggregate by summing their values. "
        f"For non-numeric columns, aggregate by taking the first observed value. "
        f"Do not apply any transformation or normalization to the aggregated values. "
        f"Group the dataframe by 'route' and aggregate all other columns according to the strategy above. "
        f"After aggregation, ensure that 'route' is present as a standard column in the final dataframe and not only as an index. "
        f"Before saving, validate that the resulting dataframe is non-empty and has unique column names. "
        f"Do not save any output if those checks fail. "
        f"Save the aggregated dataset to '{OUTPUT_DIR}/routes_summary.csv' without index. "
        f"Ensure that the dataset is loaded, processed, and saved within the same execution flow. "
        f"Avoid defining execution entry points or structures that require explicit invocation. "
        f"Assume that the code will be executed exactly as written, so all steps must run immediately. "
        f"Print only a short summary with: original merged shape, number of unique routes found, final routes_summary shape. "
        + _findings_guidance(
            "baseline_grouping",
            "Store merged_shape, unique_routes, aggregation_strategy_summary (list of numeric/non_numeric columns), routes_summary_shape. "
        )
    )


# ── Task 5: Baseline statistics ───────────────────────────────────────────────

def _build_baseline_stats_prompt():
    return (
        f"You are a baseline statistics agent responsible for enriching an already aggregated route-level dataset with baseline features for downstream anomaly detection. "

        f"Load the dataset at '{OUTPUT_DIR}/routes_summary.csv'. "
        f"Work autonomously and infer the required Python libraries. "

        f"If a findings JSON related to this dataset exists (same folder, filename ending with '_findings.json'), load it, print a short recap, and use it to guide semantic interpretation (e.g., column roles, numeric fields, warnings, distributions). "

        f"The dataset is already aggregated (e.g., per route or similar grouping). "
        f"Do not perform any additional grouping or aggregation. "

        f"First inspect the dataframe and print its shape and column names. "

        f"The dataset may contain column names in different languages (e.g., Italian) and with domain-specific terminology. "
        f"You must infer semantic meaning from column names, distributions, and findings metadata, not from fixed keywords. "

        f"Identify the main quantitative measure representing observed activity (e.g., counts, occurrences, alerts, volumes, flows, passengers, or investigated entities). "

        f"To select this measure, follow these rules: "
        f"- Prefer columns that are numeric or can be reliably converted to numeric. "
        f"- Prefer columns with meaningful variability (not constant, not mostly zero). "
        f"- Prefer columns representing aggregated quantities rather than identifiers or categories. "
        f"- Exclude columns that are clearly descriptive, categorical, identifiers, codes, dates, locations, or textual explanations. "
        f"- If findings metadata provides hints about measure columns, prioritize them. "

        f"If multiple candidates exist, choose the one that best represents the main observed phenomenon of the dataset (e.g., traffic, alerts, occurrences). "
        f"If no suitable column is found, raise an explicit error instead of forcing a weak choice. "

        f"Once the measure is selected, convert it to numeric safely, preserving missing values where appropriate. "

        f"Compute the global mean and global standard deviation of this measure across all rows. "
        f"If the standard deviation is zero or missing, set it to 1 to avoid invalid scaling. "
        f"If the mean is zero or missing, keep a safe fallback to avoid division errors. "

        f"Add the following columns: "
        f"- 'rolling_mean_alarms': global mean of the selected measure "
        f"- 'rolling_std_alarms': global standard deviation "
        f"- 'z_score': standardized deviation from the mean "
        f"- 'ratio_to_baseline': ratio between the value and the mean "

        f"Replace invalid values such as inf or -inf with 0. "

        f"Before saving, validate that the dataframe is non-empty and still consistent. "
        f"Do not save output if validation fails. "

        f"Print only a concise summary including: selected measure column, reason for selection, global mean, global std, and final shape. "
        f"Also print the top 10 rows sorted by z_score in descending order. "

        f"Save the resulting dataset to '{OUTPUT_DIR}/baseline_data.csv' without index. "

        f"Ensure all steps execute immediately without requiring explicit function calls. "

        + _findings_guidance(
            "baseline_stats",
            "Store selected_metric_name, global_mean, global_std, baseline_shape, and top_rows_by_z_score (list of records with key identifiers and computed scores)."
        )
    )


# ── Task 6: Outlier Detection ─────────────────────────────────────────────────

def _build_outlier_prompt():
    algo = DEFAULT_OUTLIER_ALGORITHM
    contam = ISOLATION_FOREST_CONTAMINATION
    neighbors = LOF_N_NEIGHBORS
    zscore_t = ZSCORE_THRESHOLD

    return (
        f"You are an outlier detection agent operating on a route-level baseline dataset. "
        f"Load the dataset at '{OUTPUT_DIR}/baseline_data.csv'. "
        f"If a shared findings JSON exists at '{FINDINGS_JSON}', load it and reuse relevant information from previous steps "
        f"(especially baseline statistics and column validation). Continue even if it is missing. "
        f"Work autonomously and infer the required Python libraries. "
        f"First inspect the dataset: print shape and column names. "
        f"Ensure that the columns representing engineered features are valid, numeric, and usable for modeling. "
        f"The expected features are 'allarmati', 'z_score', and 'ratio_to_baseline'. "
        f"Coerce invalid values to numeric form and handle non-finite values safely so that the dataset is model-ready. "
        f"Construct a feature matrix using exactly these three features, preserving row alignment with the original dataset. "
        + (
            f"Apply an Isolation Forest model using a contamination level of {contam} to detect anomalies. "
            if algo == "IsolationForest" else
            f"Apply a Local Outlier Factor model using {neighbors} neighbors and contamination {contam}. "
            if algo == "LOF" else
            f"Detect anomalies using a z-score threshold of {zscore_t} on the absolute value of z_score. "
        )
        + f"Store the result in a new boolean column named 'anomaly'. "
        f"Do not drop any columns or rows. Preserve the full dataset. "
        f"Print a short summary with: total number of rows and number of detected anomalies. "
        f"Print the top 10 anomalous rows sorted by highest anomaly severity using z_score, "
        f"displaying only 'route', 'allarmati', and 'z_score'. "
        f"Before saving, validate that the dataframe is non-empty and consistent. "
        f"If validation passes, save the dataset to '{OUTPUT_DIR}/outlier_results.csv' without index. "
        f"The dataset must be loaded, processed, and saved within the same execution flow. "
        f"All steps must run automatically without requiring explicit invocation. "
        + _findings_guidance(
            "outlier_detection",
            "Store total_rows as int, anomaly_rows as int, algorithm_used as string, "
            "feature_columns as list of strings, and top_anomalies as a list of plain Python dicts with route, allarmati, z_score. "
        )
    )


# ── Task 7: Risk Profiling ────────────────────────────────────────────────────

def _build_risk_prompt():
    mult = ALERT_RATE_MULTIPLIER
    return (
        f"Load '{OUTPUT_DIR}/outlier_results.csv' with pandas. "
        f"Import numpy as np. "
        f"Filter only rows where anomaly is True. Print how many. "
        f"If zero, save empty dataframe to '{OUTPUT_DIR}/risk_profiled.csv' and print 'No anomalies'. "
        f"Otherwise: "
        f"rule_route = anom['ratio_to_baseline'] > {mult}. "
        f"rule_zscore_high = anom['z_score'].abs() > 8. "
        f"rule_zscore_med = (anom['z_score'].abs() > 5) & (~rule_zscore_high). "
        f"conditions = [rule_route & rule_zscore_high, rule_route | rule_zscore_high, rule_zscore_med]. "
        f"choices = ['CRITICAL', 'HIGH', 'MEDIUM']. "
        f"anom['risk_level'] = np.select(conditions, choices, default='LOW'). "
        f"Print risk_level value_counts. "
        f"Save full dataframe to '{OUTPUT_DIR}/risk_profiled.csv' without index."
        + _findings_guidance(
            "risk_profiling",
            "Store anomaly_rows, risk_level_counts, rules_used (text). "
        )
    )


# ── Task 8: Report ────────────────────────────────────────────────────────────
# FIX: removed stray `s` after opening parenthesis

def _build_report_prompt():
    return (
        f"Load '{OUTPUT_DIR}/risk_profiled.csv' with pandas. "
        f"Sort by z_score descending and take top 5 rows. "
        f"Import OpenAI from openai. "
        f"Create client: client = OpenAI(base_url='{LM_STUDIO_BASE_URL}', api_key='{LM_STUDIO_API_KEY}'). "
        f"For each row, call client.chat.completions.create("
        f"model='{LM_STUDIO_MODEL}', "
        f"messages=[{{'role':'user','content':'Explain this transit anomaly in 2 sentences: ' + str(row.to_dict())}}], "
        f"max_tokens=150). "
        f"Get response text from response.choices[0].message.content. "
        f"Build a text report with header 'TRANSIT ANOMALY REPORT' and today's date. "
        f"Save report to '{OUTPUT_DIR}/anomaly_report.txt'. "
        f"Also save a JSON version with json.dump to '{OUTPUT_DIR}/anomaly_report.json'. "
        f"Print the report."
        + _findings_guidance(
            "report_generation",
            "Store top_anomalies_in_report (list), report_txt_path, report_json_path. "
        )
    )


# ── _build_tasks(): single source of truth ────────────────────────────────────
# FIX: replaces the three duplicate TASKS list definitions

def _build_tasks():
    return [
        ("data_loading_allarmi",   _build_data_prompt_1()),
        ("data_loading_tipologia", _build_data_prompt_2()),
        ("semantic_allarmi", _build_semantic_normalization_prompt(
            f"{OUTPUT_DIR}/allarmi_clean.csv",
            f"{OUTPUT_DIR}/allarmi_semantic.csv",
            "semantic_allarmi",
        )),
        ("semantic_tipologia", _build_semantic_normalization_prompt(
            f"{OUTPUT_DIR}/tipologia_clean.csv",
            f"{OUTPUT_DIR}/tipologia_semantic.csv",
            "semantic_tipologia",
        )),
        ("merge",             _build_merge_prompt()),
        ("baseline_grouping", _build_baseline_prompt()),
        ("baseline_stats",    _build_baseline_stats_prompt()),
        ("outlier_detection", _build_outlier_prompt()),
        ("risk_profiling",    _build_risk_prompt()),
        ("report_generation", _build_report_prompt()),
    ]


TASKS = _build_tasks()


# Multi-Agent System

3 agents: Supervisor (routing logic), Code_executor (direct LLM → REPL),
Validator (file check — no LLM needed).

Key design: NO tool calling. The LLM generates pure Python code,
and the code_executor runs it directly in a REPL.
- Includes rate limiting for free-tier APIs.
- Includes task caching: if a task's output file already exists, skip it.

In [34]:
import os
import time
from typing import Literal
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.types import Command
from openai import OpenAI


# Ensure output dir exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Direct LLM client (no LangChain, no tool calling)
_client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)
# NOTE: _repl is defined in the Tool REPL cell above — not re-instantiated here.

# Rate limiting
_SECONDS_BETWEEN_CALLS = 8  # 10 RPM = 1 every 6s, we use 8s for safety
_last_call_time = 0.0

# Verbose logging
VERBOSE = True  # Set to False once pipeline is stable

# Cache control
USE_CACHE = True


#  State
class AgentState(MessagesState):
    next: str
    current_task_index: int
    task_status: str
    retry_count: int


MAX_RETRIES = 4

# Maps each task name to the file it MUST produce.
EXPECTED_FILES = {
    "data_loading_allarmi":   os.path.join(OUTPUT_DIR, "allarmi_clean.csv"),
    "data_loading_tipologia": os.path.join(OUTPUT_DIR, "tipologia_clean.csv"),
    "semantic_allarmi":       os.path.join(OUTPUT_DIR, "allarmi_semantic.csv"),
    "semantic_tipologia":     os.path.join(OUTPUT_DIR, "tipologia_semantic.csv"),
    "merge":                  os.path.join(OUTPUT_DIR, "merged_data.csv"),
    "baseline_grouping":      os.path.join(OUTPUT_DIR, "routes_summary.csv"),
    "baseline_stats":         os.path.join(OUTPUT_DIR, "baseline_data.csv"),
    "outlier_detection":      os.path.join(OUTPUT_DIR, "outlier_results.csv"),
    "risk_profiling":         os.path.join(OUTPUT_DIR, "risk_profiled.csv"),
    "report_generation":      os.path.join(OUTPUT_DIR, "anomaly_report.txt"),
}


# Helper: check if task output already exists (cache)
def _task_is_cached(task_name: str) -> bool:
    if not USE_CACHE:
        return False
    filepath = EXPECTED_FILES.get(task_name, "")
    if not filepath:
        return False
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        size = os.path.getsize(filepath)
        print(f"  [cache] '{task_name}' → {os.path.basename(filepath)} already exists ({size:,} bytes), skipping.")
        return True
    return False


# Helper: rate-limited LLM call
def _call_llm(messages: list[dict], max_tokens: int = 1024) -> str:
    global _last_call_time

    elapsed = time.time() - _last_call_time
    if elapsed < _SECONDS_BETWEEN_CALLS:
        wait = _SECONDS_BETWEEN_CALLS - elapsed
        print(f"  [rate_limit] Waiting {wait:.0f}s...")
        time.sleep(wait)

    for attempt in range(3):
        try:
            _last_call_time = time.time()
            response = _client.chat.completions.create(
                model=LM_STUDIO_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.0,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                backoff = 15 * (2 ** attempt)  # 15s, 30s, 60s
                print(f"  [rate_limit] 429 received, waiting {backoff}s (attempt {attempt+1}/3)...")
                time.sleep(backoff)
            else:
                return f"LLM_ERROR: {e}"

    return "LLM_ERROR: 429 — rate limit exceeded after 3 retries"


# Helper: clean code from LLM response
def _clean_code(raw: str) -> str:
    code = raw.strip()
    if code.startswith("```"):
        lines = code.split("\n")
        lines = lines[1:]  # drop opening fence
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        code = "\n".join(lines)
    return code.strip()


# Helper: ask LLM for code and run it
def _ask_and_run(task_prompt: str, retry_context: str = "") -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Python data analyst. "
                "Reply with ONLY Python code, nothing else. "
                "No markdown, no backticks, no explanations. Just code."
            ),
        },
        {"role": "user", "content": task_prompt},
    ]

    if retry_context:
        messages.append({
            "role": "user",
            "content": f"Error: {retry_context}\nFix the code. Reply with ONLY the corrected Python code.",
        })

    raw = _call_llm(messages)

    if raw.startswith("LLM_ERROR"):
        return raw

    code = _clean_code(raw)

    if not code:
        return "LLM_ERROR: empty code response"

    if VERBOSE:
        print(f"\n  {'─'*40}")
        print(f"  GENERATED CODE:")
        print(f"  {'─'*40}")
        for i, line in enumerate(code.split("\n"), 1):
            print(f"  {i:3d} | {line}")
        print(f"  {'─'*40}\n")

    return _repl.run(code)


# Supervisor
def supervisor_node(state: AgentState) -> Command[Literal["code_executor", "validator", "__end__"]]:
    idx = state.get("current_task_index", 0)
    status = state.get("task_status", "pending")
    retries = state.get("retry_count", 0)

    if idx >= len(TASKS):
        return Command(goto="__end__", update={"next": "__end__"})

    task_name, task_prompt = TASKS[idx]

    if status == "pending" and _task_is_cached(task_name):
        next_idx = idx + 1
        if next_idx >= len(TASKS):
            return Command(goto="__end__", update={
                "current_task_index": next_idx, "task_status": "done",
            })
        return Command(
            goto="supervisor",
            update={
                "messages": [HumanMessage(content=f"Cached: {task_name}", name="supervisor")],
                "current_task_index": next_idx,
                "task_status": "pending",
                "retry_count": 0,
            },
        )

    if status in ("pending", "failed"):
        if status == "failed" and retries >= MAX_RETRIES:
            print(f"\n  SKIP '{task_name}' after {retries} retries\n")
            next_idx = idx + 1
            return Command(
                goto="__end__" if next_idx >= len(TASKS) else "supervisor",
                update={
                    "messages": [HumanMessage(content=f"Skipped {task_name}.", name="supervisor")],
                    "current_task_index": next_idx,
                    "task_status": "pending",
                    "retry_count": 0,
                },
            )
        return Command(
            goto="code_executor",
            update={
                "messages": [HumanMessage(content=task_prompt, name="supervisor")],
                "task_status": "executing",
            },
        )

    if status == "executing":
        expected = EXPECTED_FILES.get(task_name, "")
        return Command(
            goto="validator",
            update={
                "messages": [HumanMessage(content=expected, name="supervisor")],
                "task_status": "validating",
            },
        )

    if status == "validating":
        last = state["messages"][-1].content if state["messages"] else ""
        if last.startswith("APPROVED"):
            next_idx = idx + 1
            print(f"  ✓ Task '{task_name}' OK")
            if next_idx >= len(TASKS):
                return Command(goto="__end__", update={
                    "current_task_index": next_idx, "task_status": "done",
                })
            return Command(
                goto="supervisor",
                update={
                    "current_task_index": next_idx,
                    "task_status": "pending",
                    "retry_count": 0,
                },
            )
        else:
            return Command(
                goto="supervisor",
                update={"task_status": "failed", "retry_count": retries + 1},
            )

    return Command(goto="__end__", update={})


# Code Executor
def code_executor_node(state: AgentState) -> Command[Literal["supervisor"]]:
    idx = state.get("current_task_index", 0)
    retries = state.get("retry_count", 0)

    if idx >= len(TASKS):
        return Command(
            update={"messages": [HumanMessage(content="All tasks done.", name="code_executor")]},
            goto="supervisor",
        )

    task_name, task_prompt = TASKS[idx]

    retry_context = ""
    if retries > 0:
        last_msg = state["messages"][-1].content if state["messages"] else ""
        if "Error" in last_msg or "REJECTED" in last_msg:
            retry_context = last_msg

    print(f"  [code_executor] Running '{task_name}' (attempt {retries + 1})...")
    result = _ask_and_run(task_prompt, retry_context)

    if len(result) > 2000:
        result = result[:2000] + "\n... [truncated]"

    print(f"  [code_executor] Output: {result[:300]}")

    return Command(
        update={"messages": [HumanMessage(content=result, name="code_executor")]},
        goto="supervisor",
    )


# Validator (pure Python, no LLM needed)
def validator_node(state: AgentState) -> Command[Literal["supervisor"]]:
    filepath = state["messages"][-1].content if state["messages"] else ""
    filepath = filepath.strip()

    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        size = os.path.getsize(filepath)
        msg = f"APPROVED: {filepath} ({size:,} bytes)"
        print(f"  [validator] {msg}")
    else:
        if VERBOSE and os.path.isdir(OUTPUT_DIR):
            files = os.listdir(OUTPUT_DIR)
            print(f"  [validator] Files in output dir: {files}")
        msg = f"REJECTED: {filepath} not found or empty"
        print(f"  [validator] {msg}")

    return Command(
        update={"messages": [HumanMessage(content=msg, name="validator")]},
        goto="supervisor",
    )


# Graph
def build_graph():
    g = StateGraph(AgentState)
    g.add_node("supervisor", supervisor_node)
    g.add_node("code_executor", code_executor_node)
    g.add_node("validator", validator_node)
    g.add_edge(START, "supervisor")
    return g.compile()


In [35]:
PROFILE_JSON = os.path.join(OUTPUT_DIR, "column_profiles.json")

def llm_callable(prompt: str) -> str:
    response = _client.chat.completions.create(
        model=LM_STUDIO_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data profiling assistant. "
                    "Write concise, factual, business-friendly column descriptions. "
                    "Do not invent facts."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=120,
    )
    return response.choices[0].message.content.strip()


def run_pre_pipeline_column_profiling() -> dict:
    datasets_to_profile = [
        ("allarmi_raw", ALLARMI_CSV),
        ("tipologia_raw", TIPOLOGIA_CSV),
    ]

    profiles = {}

    for dataset_name, csv_path in datasets_to_profile:
        if not os.path.exists(csv_path):
            print(f"  [profiling] File not found, skipped: {csv_path}")
            continue

        print(f"  [profiling] Running column profiling for {dataset_name}...")
        profiles[dataset_name] = build_dataset_column_profiles_from_csv(
            csv_path=csv_path,
            output_json_path=PROFILE_JSON,
            llm_callable=llm_callable,
            dataset_name=dataset_name,
        )
        print(f"  [profiling] Saved profile for {dataset_name}")

    return profiles

In [36]:
print("\n" + "=" * 60)
print("  PRE-PIPELINE COLUMN PROFILING")
print("=" * 60)

profiles = run_pre_pipeline_column_profiling()

if os.path.exists(PROFILE_JSON):
    print(f"  ✓ column_profiles.json ({os.path.getsize(PROFILE_JSON):,} bytes)")
else:
    print("  ✗ column_profiles.json")



  PRE-PIPELINE COLUMN PROFILING
  [profiling] Running column profiling for allarmi_raw...
  [profiling] Saved profile for allarmi_raw
  [profiling] Running column profiling for tipologia_raw...
  [profiling] Saved profile for tipologia_raw
  ✓ column_profiles.json (67,214 bytes)


In [37]:
import json

with open("output/column_profiles.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "column_profiles": {
    "allarmi_raw": {
      "dataset_summary": {
        "dataset_name": "allarmi_raw",
        "shape": {
          "rows": 5080,
          "columns": 24
        },
        "dtypes": {
          "OCCORRENZE": "str",
          "AREOPORTO_ARRIVO": "str",
          "AREOPORTO_PARTENZA": "str",
          "ANNO_PARTENZA": "str",
          "MESE_PARTENZA": "str",
          "DATA_PARTENZA": "str",
          "DESCR_AEREOPORTO_ARR": "str",
          "DESCR_AEREOPORTO_PART": "str",
          "CITTA_ARR": "str",
          "CITTA_PARTENZA": "str",
          "CODICE_PAESE_ARR": "str",
          "CODICE_PAESE_PART": "str",
          "PAESE_ARR": "str",
          "PAESE_PART": "str",
          "ZONA": "str",
          "TOT": "str",
          "MOTIVO_ALLARME": "str",
          "note_operatore": "str",
          "flag_rischio": "str",
          "Paese Partenza": "str",
          "CODICE PAESE ARR": "str",
          "3zona": "int64",
          "paese%arr": "str",
          "tot 

In [38]:
ALGORITHM = None            # "IsolationForest", "LOF", "zscore", or None
VERBOSE_RUN = False
NO_CACHE = False
USER_QUERY = None           # e.g. "areoporto_partenza == 'FCO'" or natural language

from datetime import datetime
import os
import shutil
import pandas as pd
from langchain_core.messages import HumanMessage

# Update algorithm and cache flag
if ALGORITHM:
    DEFAULT_OUTLIER_ALGORITHM = ALGORITHM

USE_CACHE = not NO_CACHE
if NO_CACHE:
    print("  [cache] Cache disabled — all tasks will re-run.\n")

# Optional query filtering
if USER_QUERY:
    sample_df = pd.read_csv(ALLARMI_CSV, nrows=5)
    sample_df.columns = (
        sample_df.columns.str.lower().str.strip()
        .str.replace(' ', '_').str.replace('%', '_')
    )
    cols = sample_df.columns.unique()
    col_info = {
        c: sample_df[c].head(3).tolist()
        for c in cols
        if getattr(sample_df[c], "ndim", 1) == 1
    }

    _query_client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)
    system_msg = (
        "You translate user queries into pandas DataFrame .query() expressions. "
        "Reply with ONLY the query string, nothing else. No quotes around it. "
        "All column names are lowercase with underscores. "
        "Airport IATA codes: Fiumicino=FCO, Ciampino=CIA, Malpensa=MXP, Linate=LIN, "
        "Bergamo=BGY, Venezia=VCE, Bologna=BLQ, Napoli=NAP, Catania=CTA, Pisa=PSA, "
        "Bari=BRI, Palermo=PMO, Torino=TRN, Verona=VRN. "
        "Available columns and sample values: " + str(col_info)
    )
    resp = _query_client.chat.completions.create(
        model=LM_STUDIO_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": USER_QUERY},
        ],
        temperature=0.0, max_tokens=100,
    )
    pandas_query = resp.choices[0].message.content.strip()
    print(f"  User query: {USER_QUERY}")
    print(f"  LLM interpreted as: {pandas_query}")

    for csv_path in [ALLARMI_CSV, TIPOLOGIA_CSV]:
        dst = os.path.join(OUTPUT_DIR, os.path.basename(csv_path))
        shutil.copy2(csv_path, dst)
        df = pd.read_csv(dst)
        df.columns = (
            df.columns.str.lower().str.strip()
            .str.replace(' ', '_').str.replace('%', '_')
        )
        df = df.loc[:, ~df.columns.duplicated()]
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].str.strip()
        try:
            filtered = df.query(pandas_query)
            filtered.to_csv(dst, index=False)
            print(f"  Filtered {os.path.basename(csv_path)}: {len(filtered)} rows")
        except Exception as e:
            print(f"  Filter failed: {e} — keeping original data")
            df.to_csv(dst, index=False)

    # Update paths so prompt builders pick up the filtered files
    ALLARMI_CSV = os.path.join(OUTPUT_DIR, "ALLARMI.csv")
    TIPOLOGIA_CSV = os.path.join(OUTPUT_DIR, "TIPOLOGIA_VIAGGIATORE.csv")

# Rebuild TASKS once (picks up any path/algorithm changes above)
TASKS = _build_tasks()

# Run
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("  TRANSIT ANOMALY DETECTION — MULTI-AGENT (Notebook)")
print("=" * 60)
print(f"  Time:       {datetime.now().strftime('%H:%M:%S')}")
print(f"  Model:      {LM_STUDIO_MODEL}")
print(f"  Algorithm:  {DEFAULT_OUTLIER_ALGORITHM}")
print(f"  Output:     {OUTPUT_DIR}")
print("=" * 60 + "\n")

graph = build_graph()

state = {
    "messages": [HumanMessage(content="Start pipeline.")],
    "current_task_index": 0,
    "task_status": "pending",
    "next": "supervisor",
    "retry_count": 0,
}

for event in graph.stream(state, {"recursion_limit": 80}):
    for node, data in event.items():
        if node == "__end__":
            continue
        msgs = data.get("messages", [])
        if msgs:
            content = msgs[-1].content if hasattr(msgs[-1], "content") else str(msgs[-1])
            if VERBOSE_RUN:
                print(f"\n{'='*50}")
                print(f"  [{node.upper()}]")
                print(f"{'='*50}")
                if len(content) > 1000:
                    content = content[:1000] + "\n... [truncated]"
                print(content)
            else:
                line = content.split("\n")[0][:100]
                print(f"  [{node}] {line}")


print("\n" + "=" * 60)
print("  DONE")
print("=" * 60)
for f in [
    "allarmi_clean.csv",
    "tipologia_clean.csv",
    "allarmi_semantic.csv",
    "tipologia_semantic.csv",
    "merged_data.csv",
    "routes_summary.csv",
    "baseline_data.csv",
    "outlier_results.csv",
    "risk_profiled.csv",
    "anomaly_report.txt",
    "anomaly_report.json",
    "column_profiles.json",
]:
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(p):
        print(f"  ✓ {f} ({os.path.getsize(p):,} bytes)")
    else:
        print(f"  ✗ {f}")

report = os.path.join(OUTPUT_DIR, "anomaly_report.txt")
if os.path.exists(report):
    print("\n" + "=" * 60)
    print("  REPORT")
    print("=" * 60)
    with open(report, encoding="utf-8") as f:
        print(f.read())


  TRANSIT ANOMALY DETECTION — MULTI-AGENT (Notebook)
  Time:       22:10:50
  Model:      mistral-small-latest
  Algorithm:  IsolationForest
  Output:     /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output

  [supervisor] You are a data cleaning agent for tabular datasets used in downstream merge and anomaly detection ta
  [code_executor] Running 'data_loading_allarmi' (attempt 1)...


KeyboardInterrupt: 